# The Whales Stayed When the Boats Stopped

Whale-watching boats stopped in spring 2020: Monterey Bay Whale Watch ran no public trips in late
March, April and May, and resumed on 13 June
([sightings log](https://montereybaywhalewatch.com/sightings/slst2006/), read 16 October 2026).
If the public record measures whales, it should hold steady; if it measures boats, it should collapse.
The hydrophones kept recording either way, and `dim_date.inat_observers` shows whether people kept
recording wildlife at all.

The window is 15 March to 15 June (93 days, `in_spring_window`), compared across 2019, 2020 and 2021.

In [ ]:
%%sql
-- WHALES_WH runs the queries.
USE WAREHOUSE whales_wh

In [ ]:
%%sql
-- The star dbt builds lives in WHALES.MARTS.
USE SCHEMA whales.marts

## Three Springs Compared

In [ ]:
%%sql
WITH spring_days AS (
    SELECT
        dim_date.date_key,
        dim_date.year,
        dim_date.inat_observers,
        dim_date.in_hydrophone_window
    FROM dim_date
    WHERE dim_date.in_spring_window
      AND dim_date.year BETWEEN 2019 AND 2021
),

public_records_per_day AS (
    SELECT
        fct_sighting.date_key,
        COUNT(*) AS public_records
    FROM fct_sighting
    JOIN dim_location ON dim_location.location_key = fct_sighting.location_key
    JOIN dim_dataset ON dim_dataset.dataset_key = fct_sighting.dataset_key
    WHERE dim_location.in_bay
      AND dim_dataset.kind = 'public'
    GROUP BY fct_sighting.date_key
),

hydrophone_days AS (
    SELECT DISTINCT fct_sighting.date_key
    FROM fct_sighting
    JOIN dim_location ON dim_location.location_key = fct_sighting.location_key
    JOIN dim_dataset ON dim_dataset.dataset_key = fct_sighting.dataset_key
    WHERE dim_location.in_bay
      AND dim_dataset.kind = 'hydrophone'
),

spring_totals AS (
    SELECT
        spring_days.year,
        COUNT(*) AS days,
        SUM(COALESCE(public_records_per_day.public_records, 0)) AS public_records,
        COUNT_IF(spring_days.in_hydrophone_window) AS days_recording,
        COUNT(hydrophone_days.date_key) AS days_heard,
        ROUND(AVG(spring_days.inat_observers), 1) AS inat_observers_per_day
    FROM spring_days
    LEFT JOIN public_records_per_day
      ON public_records_per_day.date_key = spring_days.date_key
    LEFT JOIN hydrophone_days ON hydrophone_days.date_key = spring_days.date_key
    GROUP BY spring_days.year
)

SELECT
    TO_VARCHAR(year) AS year,
    days,
    public_records,
    days_recording,
    days_heard,
    inat_observers_per_day,
    ROUND(public_records / FIRST_VALUE(public_records) OVER (ORDER BY year) - 1, 3)
        AS public_records_change_from_2019,
    ROUND(
        public_records / AVG(IFF(year IN (2019, 2021), public_records, NULL)) OVER () - 1,
        3
    ) AS public_records_change_from_2019_2021_average,
    ROUND(days_heard / days_recording, 2) AS share_of_days_heard
FROM spring_totals
ORDER BY year

## Every Spring Since 2010
Public sightings in the same window each year, to show how far an ordinary spring moves.

In [ ]:
%%sql
SELECT
    TO_VARCHAR(dim_date.year) AS year,
    COUNT(*) AS public_records
FROM fct_sighting
JOIN dim_location ON dim_location.location_key = fct_sighting.location_key
JOIN dim_date ON dim_date.date_key = fct_sighting.date_key
JOIN dim_dataset ON dim_dataset.dataset_key = fct_sighting.dataset_key
WHERE dim_location.in_bay
  AND dim_dataset.kind = 'public'
  AND dim_date.in_spring_window
  AND dim_date.year BETWEEN 2010 AND 2025
GROUP BY dim_date.year
ORDER BY dim_date.year

## Result
In spring 2020 public sightings fell **79%** against spring 2019 (1,253 to 266), and 77% against the
average of 2019 and 2021. Every other spring from 2018 to 2025 had at least 1,070, and 266 is the lowest
since 2013. The hydrophones heard whales on 92 of the 93 days in their recording window, against 60 of 93
in 2019 and 80 of 90 in 2021, and iNaturalist users kept recording wildlife around the bay.

## Limits
* Beaches and harbors closed as well as the boats.
* Ordinary springs swing widely too (2,560 in 2022, 1,191 in 2023).
* Days heard show whales were present, not how many. A quieter ocean may have helped the hydrophones hear,
  and a day they were out of the water counts as not heard.
* iNaturalist was growing anyway (67.6 observers a day in spring 2021).